In [1]:

import pandas as pd

# Three ESG incident messages used in the Gemini experiments
messages = {
    "Water Leak": (
        "A water pipe has burst in Building C, Level 1. "
        "Water is reaching exposed electrical cables, and employees "
        "are standing nearby. The area has not been isolated."
    ),
    "Energy Waste": (
        "The lights and air conditioning in Building B have been left "
        "running overnight for the past three days, even though "
        "the offices are empty. There is no immediate safety danger, "
        "but this is wasting electricity."
    ),
    "Accessibility": (
        "The wheelchair-accessible entrance to Building A has been "
        "blocked by construction equipment for two days. Employees "
        "who use wheelchairs cannot independently access the building, "
        "and no alternative accessible entrance has been provided."
    )
}

# Rule-based classifier: no LLM is used
def rule_based_classifier(message):
    text = message.lower()

    if "wheelchair" in text or "accessible entrance" in text:
        category = "ACCESSIBILITY"
        team = "Accessibility"

    elif "water pipe" in text or "water leak" in text:
        category = "WATER_LEAK"
        team = "Facilities"

    elif "air conditioning" in text or "electricity" in text:
        category = "ENERGY_WASTE"
        team = "Facilities"

    elif "recycling" in text or "waste disposal" in text:
        category = "WASTE_RECYCLING"
        team = "Sustainability"

    else:
        category = "UNKNOWN"
        team = "Human Review"

    # Prioritise obvious safety and accessibility risks
    if category == "WATER_LEAK" and "electrical cables" in text:
        urgency = "CRITICAL"
        team = "Health and Safety"

    elif category == "ACCESSIBILITY" and "blocked" in text:
        urgency = "HIGH"

    elif category == "ENERGY_WASTE" and "overnight" in text:
        urgency = "MEDIUM"

    elif category == "UNKNOWN":
        urgency = "LOW"

    else:
        urgency = "MEDIUM"

    followup = "N" if category == "UNKNOWN" else "Y"

    return {
        "category": category,
        "urgency": urgency,
        "followup": followup,
        "team": team
    }

# Gemini results transcribed from the clean test screenshots
gemini_results = {
    "Water Leak": ("WATER_LEAK", "CRITICAL", "Y", "Health and Safety"),
    "Energy Waste": ("ENERGY_WASTE", "LOW", "Y", "Facilities"),
    "Accessibility": ("ACCESSIBILITY", "HIGH", "Y", "Accessibility")
}

# Generate an independent baseline and compare classifications
results = []

for scenario, message in messages.items():
    baseline = rule_based_classifier(message)
    llm = gemini_results[scenario]

    results.append({
        "Scenario": scenario,
        "Rule Category": baseline["category"],
        "Gemini Category": llm[0],
        "Rule Urgency": baseline["urgency"],
        "Gemini Urgency": llm[1],
        "Category Match": baseline["category"] == llm[0],
        "Urgency Match": baseline["urgency"] == llm[1],
        "Rule Team": baseline["team"],
        "Gemini Team": llm[3]
    })

comparison = pd.DataFrame(results)

print("RULE-BASED CLASSIFIER VS GEMINI LLM")
display(comparison)

print("\nCATEGORY AGREEMENT:",
      f"{comparison['Category Match'].sum()}/3")

print("URGENCY AGREEMENT:",
      f"{comparison['Urgency Match'].sum()}/3")


RULE-BASED CLASSIFIER VS GEMINI LLM


,Scenario,Rule Category,Gemini Category,Rule Urgency,Gemini Urgency,Category Match,Urgency Match,Rule Team,Gemini Team
0,Water Leak,WATER_LEAK,WATER_LEAK,CRITICAL,CRITICAL,True,True,Health and Safety,Health and Safety
1,Energy Waste,ENERGY_WASTE,ENERGY_WASTE,MEDIUM,LOW,True,False,Facilities,Facilities
2,Accessibility,ACCESSIBILITY,ACCESSIBILITY,HIGH,HIGH,True,True,Accessibility,Accessibility



CATEGORY AGREEMENT: 3/3
URGENCY AGREEMENT: 2/3
